In [4]:
import random

NETWORK_LINKS = {
    ('A', 'B'): 100, ('B', 'A'): 100,
    ('A', 'C'): 50,  ('C', 'A'): 50,
    ('B', 'C'): 100, ('C', 'B'): 100,
    ('B', 'D'): 60,  ('D', 'B'): 60,
    ('C', 'D'): 100, ('D', 'C'): 100
}

FLOWS = [
    {'id': 1, 'src': 'A', 'dst': 'D', 'demand': 40},
    {'id': 2, 'src': 'A', 'dst': 'C', 'demand': 30},
    {'id': 3, 'src': 'B', 'dst': 'D', 'demand': 50}
]

AVAILABLE_PATHS = {
    1: [['A', 'B', 'D'], ['A', 'C', 'D'], ['A', 'B', 'C', 'D']],
    2: [['A', 'C'], ['A', 'B', 'C']],
    3: [['B', 'D'], ['B', 'C', 'D'], ['B', 'A', 'C', 'D']]
}


POPULATION_SIZE = 20
GENERATIONS = 50
MUTATION_RATE = 0.2

class TrafficManagerGA:
    def __init__(self):
        self.population = self._initialize_population()

    def _initialize_population(self):
        population = []
        for _ in range(POPULATION_SIZE):
            chromosome = [random.randint(0, len(AVAILABLE_PATHS[f['id']]) - 1) for f in FLOWS]
            population.append(chromosome)
        return population

    def evaluate_fitness(self, chromosome):
        link_loads = {link: 0 for link in NETWORK_LINKS}
        penalty = 0

        for i, flow in enumerate(FLOWS):
            path_idx = chromosome[i]
            path = AVAILABLE_PATHS[flow['id']][path_idx]
            demand = flow['demand']

            for j in range(len(path) - 1):
                edge = (path[j], path[j+1])
                if edge in link_loads:
                    link_loads[edge] += demand
                else:
                    penalty += 1000

        max_utilization = 0
        for link, load in link_loads.items():
            capacity = NETWORK_LINKS[link]
            utilization = load / capacity
            if utilization > max_utilization:
                max_utilization = utilization
            if load > capacity:
                penalty += (load - capacity) * 100

        return max_utilization + penalty

    def crossover(self, parent1, parent2):
        crossover_point = random.randint(1, len(FLOWS) - 1)
        child1 = parent1[:crossover_point] + parent2[crossover_point:]
        child2 = parent2[:crossover_point] + parent1[crossover_point:]
        return child1, child2

    def mutate(self, chromosome):
        if random.random() < MUTATION_RATE:
            flow_idx = random.randint(0, len(FLOWS) - 1)
            flow_id = FLOWS[flow_idx]['id']
            new_path_idx = random.randint(0, len(AVAILABLE_PATHS[flow_id]) - 1)
            chromosome[flow_idx] = new_path_idx
        return chromosome

    def select_parent(self, fitness_scores):
        tournament = random.sample(list(enumerate(fitness_scores)), 3)
        best_idx = min(tournament, key=lambda x: x[1])[0]
        return self.population[best_idx]

    def run(self):
        best_chromosome = None
        best_fitness = float('inf')

        for generation in range(GENERATIONS):
            fitness_scores = [self.evaluate_fitness(chrom) for chrom in self.population]

            current_best_idx = min(range(len(fitness_scores)), key=lambda i: fitness_scores[i])
            if fitness_scores[current_best_idx] < best_fitness:
                best_fitness = fitness_scores[current_best_idx]
                best_chromosome = self.population[current_best_idx]

            new_population = [best_chromosome]

            while len(new_population) < POPULATION_SIZE:
                p1 = self.select_parent(fitness_scores)
                p2 = self.select_parent(fitness_scores)

                c1, c2 = self.crossover(p1, p2)
                new_population.extend([self.mutate(c1), self.mutate(c2)])

            self.population = new_population[:POPULATION_SIZE]

            if generation % 10 == 0:
                print(f"Generation {generation} | Best Fitness (Max Utilization): {best_fitness:.2f}")

        return best_chromosome, best_fitness

if __name__ == "__main__":
    ga = TrafficManagerGA()
    best_routing, final_fitness = ga.run()

    print("\n--- Final Optimized Routing Table ---")
    for i, flow in enumerate(FLOWS):
        chosen_path = AVAILABLE_PATHS[flow['id']][best_routing[i]]
        print(f"Flow {flow['id']} ({flow['src']} -> {flow['dst']}, Demand: {flow['demand']} Mbps) "
              f"routed via: {' -> '.join(chosen_path)}")

Generation 0 | Best Fitness (Max Utilization): 0.67
Generation 10 | Best Fitness (Max Utilization): 0.67
Generation 20 | Best Fitness (Max Utilization): 0.67
Generation 30 | Best Fitness (Max Utilization): 0.67
Generation 40 | Best Fitness (Max Utilization): 0.67

--- Final Optimized Routing Table ---
Flow 1 (A -> D, Demand: 40 Mbps) routed via: A -> B -> D
Flow 2 (A -> C, Demand: 30 Mbps) routed via: A -> C
Flow 3 (B -> D, Demand: 50 Mbps) routed via: B -> C -> D
